# 第 11 章：别让 Lead 亲自搬运所有原始材料（概念实验与工程迁移）

按正文顺序完成每个实验：先写预测，再运行代码，阅读输出，最后修改一个变量。

概念实验不会预先导入 Mini DeerFlow；进入“工程迁移”标签后，才把同一机制放回项目。

## 实验 1：原始 specialist 输出直接进入 Lead 历史

`concept` · `failure` · `delegation-boundary`

**运行前先预测**：两段 specialist 原始输出共有 2600 字符。最终综合时，Lead 是否仍要重新读取它们？Secret 是否也进入同一输入？

> 先在这里写下你的判断，再执行下一个代码单元。

In [1]:
research_raw = "R" * 1200
coding_raw = "C" * 1400
auth_token = "sk-live-course"

lead_history = [
    {"role": "user", "content": "比较 reducer 的语义与 Python 实现"},
    {"role": "tool", "content": research_raw},
    {"role": "tool", "content": coding_raw},
    {"role": "system", "content": f"auth_token={auth_token}"},
]
synthesis_input = "\n".join(message["content"] for message in lead_history)

print("message_count =", len(lead_history))
print("specialist_raw_chars =", len(research_raw) + len(coding_raw))
print("secret_in_synthesis_input =", auth_token in synthesis_input)
print("raw_results_still_in_history =", research_raw in synthesis_input and coding_raw in synthesis_input)


message_count = 4
specialist_raw_chars = 2600
secret_in_synthesis_input = True
raw_results_still_in_history = True


**发生了什么**：函数完成了分工，却没有建立委派边界。原始结果与 Secret 都进入 Lead 的长期输入，后续每轮综合都要再次承担读取、存储和误用成本。

**动手修改**：把每段原始输出扩大到 10000 字符。说明为什么“换更大上下文窗口”只会推迟失败，而不会建立权限和生命周期边界。

## 实验 2：用请求与结果协议切断原始材料

`concept` · `repair` · `delegation-boundary`

**运行前先预测**：specialist 只收到任务描述和 locale，Lead 只保存 32 字符摘要与 ArtifactRef。原始材料和 auth_token 还会进入结果吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [2]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class TaskRequest(BaseModel):
    model_config = ConfigDict(frozen=True)

    task_id: str
    specialist: Literal["research", "coding"]
    prompt: str = Field(min_length=1, max_length=200)
    context: dict[str, str] = Field(default_factory=dict)


class TaskResult(BaseModel):
    model_config = ConfigDict(frozen=True)

    task_id: str
    specialist: str
    status: Literal["completed", "failed", "timed_out", "output_too_large"]
    summary: str = Field(max_length=32)
    artifact_paths: list[str] = Field(default_factory=list, max_length=2)


request = TaskRequest(
    task_id="research-001",
    specialist="research",
    prompt="比较 reducer 语义",
    context={"locale": "zh-CN"},
)
result = TaskResult(
    task_id=request.task_id,
    specialist=request.specialist,
    status="completed",
    summary="并行更新必须通过 reducer 合并",
    artifact_paths=["artifacts/reducer-notes.md"],
)
serialized = result.model_dump_json()

print("request_context_keys =", sorted(request.context))
print("result_fields =", sorted(result.model_dump()))
print("secret_in_result =", auth_token in serialized)
print("raw_research_in_result =", research_raw in serialized)
print("artifact_count =", len(result.artifact_paths))


request_context_keys = ['locale']
result_fields = ['artifact_paths', 'specialist', 'status', 'summary', 'task_id']
secret_in_result = False
raw_research_in_result = False
artifact_count = 1


**发生了什么**：Subagent 的价值在于稳定的输入、执行和返回边界。完整材料落到 Artifact repository，Lead 只持有综合所需的摘要与引用。

**动手修改**：尝试构造 33 字符摘要和 3 个 artifact。记录 Pydantic 在模型调用前拒绝了哪两个越界输入。

## 实验 3：深拷贝仍然复制了完整主上下文

`concept` · `failure` · `context-projection`

**运行前先预测**：`deepcopy` 会创建新对象。它是否也会自动删除 messages、auth_token 和 Lead 的内部笔记？

> 先在这里写下你的判断，再执行下一个代码单元。

In [3]:
from copy import deepcopy


parent_context = {
    "user_id": "learner-11",
    "locale": "zh-CN",
    "messages": ["主会话消息 1", "主会话消息 2"],
    "auth_token": "never-forward",
    "internal_notes": "Lead 的私有规划",
}
naive_child_context = deepcopy(parent_context)

print("child_keys =", sorted(naive_child_context))
print("copied_is_new_object =", naive_child_context is not parent_context)
print("message_count =", len(naive_child_context["messages"]))
print("secret_visible =", naive_child_context["auth_token"] == "never-forward")


child_keys = ['auth_token', 'internal_notes', 'locale', 'messages', 'user_id']
copied_is_new_object = True
message_count = 2
secret_visible = True


**发生了什么**：新字典与父字典是两个对象，但内容仍然相同。权限边界不能依赖 specialist “自觉不用”已经收到的数据。

**动手修改**：把 messages 改成 100 条，再观察 child 的键和值。说明深拷贝为何还会增加内存成本。

## 实验 4：从空上下文开始，只投影允许字段

`concept` · `repair` · `context-projection`

**运行前先预测**：allowlist 只有 user_id 和 locale。生成的新 invocation 中还会出现 messages、internal_notes 或 auth_token 吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class SpecialistInvocation:
    task_id: str
    prompt: str
    context: dict[str, str]


def project_context(source: dict[str, object], allowed: frozenset[str]) -> dict[str, str]:
    return {
        key: str(source[key])
        for key in allowed
        if key in source and isinstance(source[key], str)
    }


allowed_fields = frozenset({"user_id", "locale"})
invocation = SpecialistInvocation(
    task_id="context-001",
    prompt="只比较公开接口",
    context=project_context(parent_context, allowed_fields),
)
rendered_invocation = repr(invocation)

print("projected_keys =", sorted(invocation.context))
print("messages_visible =", "messages" in invocation.context)
print("internal_notes_visible =", "internal_notes" in invocation.context)
print("secret_visible =", "never-forward" in rendered_invocation)


projected_keys = ['locale', 'user_id']
messages_visible = False
internal_notes_visible = False
secret_visible = False


**发生了什么**：投影从空对象开始，只复制 allowlist 中的字段。它建立了数据最小化边界；真正调用工具时，服务端仍要重新检查身份与权限。

**动手修改**：把 auth_token 加入 allowlist，观察它确实会泄漏。然后在 `project_context` 中拒绝以 token、secret 或 key 结尾的字段名。

## 实验 5：Command 在一次路由中同时更新 State 与选择节点

`concept` · `baseline` · `router-command`

**运行前先预测**：输入包含“Python 修复”。router 会运行 research 和 coding 两个节点，还是只进入 coding？

> 先在这里写下你的判断，再执行下一个代码单元。

In [5]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


class RouterState(TypedDict, total=False):
    query: str
    route: Literal["research", "coding"]
    answer: str
    trace: Annotated[list[str], operator.add]


def route_once(state: RouterState) -> Command[Literal["research", "coding"]]:
    selected: Literal["research", "coding"] = (
        "coding" if "Python" in state["query"] or "修复" in state["query"] else "research"
    )
    return Command(goto=selected, update={"route": selected, "trace": [f"router:{selected}"]})


def research(_: RouterState) -> dict[str, object]:
    return {"answer": "research result", "trace": ["research"]}


def coding(_: RouterState) -> dict[str, object]:
    return {"answer": "coding result", "trace": ["coding"]}


builder = StateGraph(RouterState)
builder.add_node("router", route_once)
builder.add_node("research", research)
builder.add_node("coding", coding)
builder.add_edge(START, "router")
builder.add_edge("research", END)
builder.add_edge("coding", END)
router_graph = builder.compile()
router_result = router_graph.invoke({"query": "修复 Python reducer"})

print("selected_route =", router_result["route"])
print("trace =", router_result["trace"])
print("specialist_runs =", len(router_result["trace"]) - 1)


selected_route = coding
trace = ['router:coding', 'coding']
specialist_runs = 1


**发生了什么**：`Command(update=..., goto=...)` 一边记录路由结果，一边只选择一个后继节点。Router 适合分类后进入固定分支，不要求 Lead 等结果回来再规划。

**动手修改**：把 query 改成“比较 checkpoint 文档”。先预测 route，再运行；确认 trace 中仍只有一个 specialist。

## 实验 6：Send 为每个目标创建独立输入，并通过 reducer 汇总

`concept` · `baseline` · `router-send`

**运行前先预测**：routes 同时包含 research 和 coding。两个 worker 返回的列表会覆盖，还是由 reducer 合并？

> 先在这里写下你的判断，再执行下一个代码单元。

In [6]:
from langgraph.types import Send


class FanoutState(TypedDict, total=False):
    query: str
    routes: list[Literal["research", "coding"]]
    specialist: Literal["research", "coding"]
    results: Annotated[list[str], operator.add]
    answer: str


def fan_out(state: FanoutState) -> list[Send]:
    return [
        Send("specialist", {"query": state["query"], "specialist": name})
        for name in state["routes"]
    ]


def run_branch(state: FanoutState) -> dict[str, list[str]]:
    return {"results": [f"{state['specialist']}:{state['query']}"]}


def synthesize(state: FanoutState) -> dict[str, str]:
    names = sorted(item.split(":", 1)[0] for item in state["results"])
    return {"answer": "+".join(names)}


fanout_builder = StateGraph(FanoutState)
fanout_builder.add_node("specialist", run_branch)
fanout_builder.add_node("synthesize", synthesize)
fanout_builder.add_conditional_edges(START, fan_out, ["specialist"])
fanout_builder.add_edge("specialist", "synthesize")
fanout_builder.add_edge("synthesize", END)
fanout_graph = fanout_builder.compile()
fanout_result = fanout_graph.invoke(
    {"query": "比较语义与实现", "routes": ["research", "coding"]}
)

print("result_count =", len(fanout_result["results"]))
print("result_names =", sorted(item.split(":", 1)[0] for item in fanout_result["results"]))
print("answer =", fanout_result["answer"])


result_count = 2
result_names = ['coding', 'research']
answer = coding+research


**发生了什么**：`Send` 为每个分支构造输入，`Annotated[list, operator.add]` 在 fan-in 时合并结果。并行只改变调度，传入哪些上下文字段仍由应用决定。

**动手修改**：删除 results 的 reducer，再运行两个分支。记录 LangGraph 为什么拒绝同一 superstep 对同一 key 的并发更新。

## 实验 7：active_agent 让后续请求绕过 triage

`concept` · `baseline` · `handoff-ownership`

**运行前先预测**：第一轮选中 coding 后，第二轮带着 active_agent=coding 再进入图。triage 会再次运行吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [7]:
class HandoffState(TypedDict, total=False):
    request: str
    active_agent: Literal["triage", "research", "coding"]
    answer: str
    trace: Annotated[list[str], operator.add]


def enter(state: HandoffState) -> Command[Literal["triage", "research", "coding"]]:
    owner = state.get("active_agent")
    return Command(goto=owner if owner in {"research", "coding"} else "triage")


def triage(state: HandoffState) -> Command[Literal["research", "coding"]]:
    selected: Literal["research", "coding"] = (
        "coding" if "代码" in state["request"] else "research"
    )
    return Command(goto=selected, update={"active_agent": selected, "trace": [f"triage->{selected}"]})


def specialist_answer(state: HandoffState) -> dict[str, object]:
    owner = state["active_agent"]
    return {"answer": f"{owner} owns this turn", "trace": [f"{owner}:answered"]}


handoff_builder = StateGraph(HandoffState)
handoff_builder.add_node("enter", enter)
handoff_builder.add_node("triage", triage)
handoff_builder.add_node("research", specialist_answer)
handoff_builder.add_node("coding", specialist_answer)
handoff_builder.add_edge(START, "enter")
handoff_builder.add_edge("research", END)
handoff_builder.add_edge("coding", END)
handoff_graph = handoff_builder.compile()

first_turn = handoff_graph.invoke({"request": "检查代码接口"})
second_turn = handoff_graph.invoke(
    {"request": "继续解释", "active_agent": first_turn["active_agent"]}
)

print("first_owner =", first_turn["active_agent"])
print("first_trace =", first_turn["trace"])
print("second_trace =", second_turn["trace"])
print("triage_ran_again =", any("triage" in item for item in second_turn["trace"]))


first_owner = coding
first_trace = ['triage->coding', 'coding:answered']
second_trace = ['coding:answered']
triage_ran_again = False


**发生了什么**：Handoff 把 active owner 写进 State。目标 Agent 不再只返回一次结果，而是接管后续对话。若最终必须回到 Lead 统一审阅和综合，这种所有权就不合适。

**动手修改**：去掉第二轮的 active_agent。观察 triage 再次运行，并解释持久化会话中该字段应该由谁保存。

## 实验 8：共享 Schema 的子图能看见父图传入的字段

`concept` · `contrast` · `subgraph-boundary`

**运行前先预测**：父图把完整 State 传给共享 Schema 子图。子节点能否看见 messages 和 auth_token？

> 先在这里写下你的判断，再执行下一个代码单元。

In [8]:
class SharedState(TypedDict, total=False):
    query: str
    messages: list[str]
    auth_token: str
    child_observed_keys: list[str]
    secret_visible: bool


def inspect_shared_state(state: SharedState) -> dict[str, object]:
    return {
        "child_observed_keys": sorted(state),
        "secret_visible": "auth_token" in state,
    }


child_builder = StateGraph(SharedState)
child_builder.add_node("inspect", inspect_shared_state)
child_builder.add_edge(START, "inspect")
child_builder.add_edge("inspect", END)
child_graph = child_builder.compile()

parent_builder = StateGraph(SharedState)
parent_builder.add_node("specialist_subgraph", child_graph)
parent_builder.add_edge(START, "specialist_subgraph")
parent_builder.add_edge("specialist_subgraph", END)
parent_graph = parent_builder.compile()
shared_result = parent_graph.invoke(
    {"query": "检查边界", "messages": ["主历史"], "auth_token": "hidden"}
)

print("child_observed_keys =", shared_result["child_observed_keys"])
print("messages_visible =", "messages" in shared_result["child_observed_keys"])
print("secret_visible =", shared_result["secret_visible"])


child_observed_keys = ['auth_token', 'messages', 'query']
messages_visible = True
secret_visible = True


**发生了什么**：Subgraph 提供嵌套拓扑、复用和 checkpoint 可见性，但不会自动缩小上下文。需要隔离时，应使用不同 Schema 或 adapter 显式投影输入与输出。

**动手修改**：在父图和子图之间增加 adapter，只传 query。不要只在子节点中忽略字段；要让未授权数据根本不进入子图输入。

## 实验 9：每次委派创建新的 invocation，只返回稳定结果

`concept` · `baseline` · `ephemeral-subagent`

**运行前先预测**：连续调用同一个 research specialist 两次。第二次 invocation 是否能看到第一次的 prompt？

> 先在这里写下你的判断，再执行下一个代码单元。

In [9]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class MinimalTask:
    task_id: str
    specialist: str
    prompt: str


@dataclass(frozen=True)
class MinimalResult:
    task_id: str
    specialist: str
    status: str
    summary: str
    observed_prompts: tuple[str, ...] = field(default_factory=tuple)


def invoke_ephemeral(task: MinimalTask) -> MinimalResult:
    fresh_messages = [task.prompt]
    return MinimalResult(
        task_id=task.task_id,
        specialist=task.specialist,
        status="completed",
        summary=f"摘要:{task.prompt}",
        observed_prompts=tuple(fresh_messages),
    )


first_result = invoke_ephemeral(MinimalTask("task-1", "research", "解释 reducer"))
second_result = invoke_ephemeral(MinimalTask("task-2", "research", "解释 checkpoint"))

print("first_observed =", first_result.observed_prompts)
print("second_observed =", second_result.observed_prompts)
print("first_prompt_leaked_to_second =", "解释 reducer" in second_result.observed_prompts)
print("lead_regains_control =", first_result.status == second_result.status == "completed")


first_observed = ('解释 reducer',)
second_observed = ('解释 checkpoint',)
first_prompt_leaked_to_second = False
lead_regains_control = True


**发生了什么**：临时 Subagent 的生命周期只有一笔委派。Lead 收到结果后继续决定下一步；若 specialist 要长期直接服务用户，应选择 Handoff。

**动手修改**：故意把 `fresh_messages` 提升为全局 list。运行两次后观察串线，并解释为什么“给每个 specialist 一个永久历史”改变了产品语义。

## 实验 10：gather 会一次启动所有 coroutine

`concept` · `failure` · `concurrency-control`

**运行前先预测**：同时提交 4 个任务，没有 Semaphore。实际执行峰值是 1、2，还是 4？

> 先在这里写下你的判断，再执行下一个代码单元。

In [10]:
import asyncio


unbounded_counter = {"active": 0, "peak": 0}


async def unbounded_worker(name: str) -> str:
    unbounded_counter["active"] += 1
    unbounded_counter["peak"] = max(
        unbounded_counter["peak"], unbounded_counter["active"]
    )
    await asyncio.sleep(0.01)
    unbounded_counter["active"] -= 1
    return f"done:{name}"


async def run_unbounded() -> list[str]:
    return list(await asyncio.gather(*(unbounded_worker(str(i)) for i in range(4))))


unbounded_results = await run_unbounded()
print("submitted =", len(unbounded_results))
print("peak_concurrency =", unbounded_counter["peak"])
print("all_completed =", all(item.startswith("done:") for item in unbounded_results))


submitted = 4
peak_concurrency = 4
all_completed = True


**发生了什么**：`gather` 负责等待和聚合，不负责资源配额。模型同轮产生多少 tool calls，执行器就可能同时启动多少后端请求。
Notebook 已有运行中的事件循环，所以本章直接 `await`。把同一段逻辑移到 `.py` 脚本时，才在最外层使用 `asyncio.run(run_unbounded())`。

**动手修改**：把任务数改为 20。即使本地仍能完成，也要说明供应商 rate limit、连接池和 Sandbox 资源会怎样放大风险。

## 实验 11：Semaphore 把峰值锁在执行入口

`concept` · `repair` · `concurrency-control`

**运行前先预测**：提交数量仍是 4，Semaphore 容量是 2。结果数量和峰值分别是多少？

> 先在这里写下你的判断，再执行下一个代码单元。

In [11]:
limited_counter = {"active": 0, "peak": 0}


async def run_limited() -> list[str]:
    semaphore = asyncio.Semaphore(2)

    async def limited_worker(name: str) -> str:
        async with semaphore:
            limited_counter["active"] += 1
            limited_counter["peak"] = max(
                limited_counter["peak"], limited_counter["active"]
            )
            await asyncio.sleep(0.01)
            limited_counter["active"] -= 1
            return f"done:{name}"

    return list(await asyncio.gather(*(limited_worker(str(i)) for i in range(4))))


limited_results = await run_limited()
print("submitted =", len(limited_results))
print("peak_concurrency =", limited_counter["peak"])
print("result_order =", limited_results)


submitted = 4
peak_concurrency = 2
result_order = ['done:0', 'done:1', 'done:2', 'done:3']


**发生了什么**：Semaphore 位于真实执行入口，不依赖模型遵守提示。它只限制同时运行数，不会限制队列长度、CPU、子进程或外部副作用。

**动手修改**：把容量改为 1 和 4，分别观察峰值。再写下生产系统还需要的队列长度与租户配额。

## 实验 12：裸 gather 抛出异常，调用方拿不到业务结果列表

`concept` · `failure` · `partial-failure`

**运行前先预测**：fast 已先完成，boom 随后抛错，slow 仍在等待。调用方能否拿到包含 fast 的结构化结果列表？

> 先在这里写下你的判断，再执行下一个代码单元。

In [12]:
async def run_naive_batch() -> tuple[str, list[str], int, bool]:
    side_effects: list[str] = []

    async def worker(name: str, delay: float, should_fail: bool = False) -> str:
        await asyncio.sleep(delay)
        if should_fail:
            raise RuntimeError("provider unavailable")
        side_effects.append(name)
        return f"ok:{name}"

    tasks = [
        asyncio.create_task(worker("fast", 0.001)),
        asyncio.create_task(worker("boom", 0.01, True)),
        asyncio.create_task(worker("slow", 0.05)),
    ]
    try:
        visible_results = list(await asyncio.gather(*tasks))
        error_type = "none"
    except RuntimeError as error:
        error_type = type(error).__name__
        visible_results = []
    slow_was_pending = not tasks[2].done()
    for task in tasks:
        if not task.done():
            task.cancel()
    await asyncio.gather(*tasks, return_exceptions=True)
    return error_type, side_effects, len(visible_results), slow_was_pending


batch_error, completed_side_effects, visible_count, slow_pending = await run_naive_batch()
print("batch_error =", batch_error)
print("completed_side_effects =", completed_side_effects)
print("visible_result_count =", visible_count)
print("slow_was_pending =", slow_pending)


batch_error = RuntimeError
completed_side_effects = ['fast']
visible_result_count = 0
slow_was_pending = True


**发生了什么**：fast 已经成功，裸异常却越过批量边界，调用方只得到一个 exception。没有逐任务结果协议，Lead 无法利用部分证据，也分不清失败与超时。

**动手修改**：给 gather 加 `return_exceptions=True`。观察列表形状改善了什么，再解释为什么裸 Exception 仍不是稳定业务协议。

## 实验 13：每个任务单独归一化为 completed、failed 或 timed_out

`concept` · `repair` · `partial-failure`

**运行前先预测**：三个请求顺序是 success、failure、timeout。并发执行后，结果协议是否仍保持这个输入顺序？

> 先在这里写下你的判断，再执行下一个代码单元。

In [13]:
async def controlled_worker(name: str) -> str:
    if name == "failure":
        raise RuntimeError("provider unavailable")
    if name == "timeout":
        await asyncio.sleep(0.05)
    return f"ok:{name}"


async def safe_dispatch(name: str) -> dict[str, str]:
    try:
        value = await asyncio.wait_for(controlled_worker(name), timeout=0.01)
    except TimeoutError:
        return {"name": name, "status": "timed_out", "value": ""}
    except Exception as error:
        return {"name": name, "status": "failed", "value": type(error).__name__}
    return {"name": name, "status": "completed", "value": value}


async def run_safe_batch() -> list[dict[str, str]]:
    names = ["success", "failure", "timeout"]
    return list(await asyncio.gather(*(safe_dispatch(name) for name in names)))


safe_results = await run_safe_batch()
print("names =", [item["name"] for item in safe_results])
print("statuses =", [item["status"] for item in safe_results])
print("success_value =", safe_results[0]["value"])
print("failure_value =", safe_results[1]["value"])


names = ['success', 'failure', 'timeout']
statuses = ['completed', 'failed', 'timed_out']
success_value = ok:success
failure_value = RuntimeError


**发生了什么**：每个 task 都有自己的 failure boundary，批量层只聚合稳定结果。`timed_out` 表示执行预算耗尽，和业务明确失败的 `failed` 是两种终态。

**动手修改**：把 timeout 调到 0.1 秒。观察第三个结果转为 completed，并说明 event-loop timeout 为什么不能证明阻塞式外部进程已被强杀。

## 实验 14：完整摘要和全部 ArtifactRef 被序列化进消息

`concept` · `failure` · `output-budget`

**运行前先预测**：summary 有 160 字符、artifact 有 5 个。缺少预算时，消息里会保留多少？

> 先在这里写下你的判断，再执行下一个代码单元。

In [14]:
import json


full_summary = "证" * 160
full_artifacts = [f"artifacts/{index}.md" for index in range(5)]
unsafe_payload = {
    "status": "completed",
    "summary": full_summary,
    "artifacts": full_artifacts,
}
unsafe_tool_message = json.dumps(unsafe_payload, ensure_ascii=False)

print("summary_chars_in_message =", len(unsafe_payload["summary"]))
print("artifact_refs_in_message =", len(unsafe_payload["artifacts"]))
print("full_summary_present =", full_summary in unsafe_tool_message)
print("message_chars =", len(unsafe_tool_message))


summary_chars_in_message = 160
artifact_refs_in_message = 5
full_summary_present = True
message_chars = 303


**发生了什么**：状态虽然写着 completed，消息、checkpoint 和 trace 却都复制了完整输出。长度问题被推迟到下一轮模型调用，没有在结果产生处暴露。

**动手修改**：把 summary 扩大到 10000 字符。比较 ToolMessage、checkpoint 和 trace 可能产生的重复存储。

## 实验 15：返回有界 preview、原始长度和 digest

`concept` · `repair` · `output-budget`

**运行前先预测**：预算允许 32 字符和 2 个引用。完整结果能否继续直接进入 Lead？digest 能否恢复原文？

> 先在这里写下你的判断，再执行下一个代码单元。

In [15]:
import hashlib


digest_payload = json.dumps(
    {"summary": full_summary, "artifacts": full_artifacts},
    ensure_ascii=False,
    sort_keys=True,
)
bounded_payload = {
    "status": "output_too_large",
    "summary": full_summary[:32],
    "artifacts": full_artifacts[:2],
    "output_chars": len(full_summary),
    "output_sha256": hashlib.sha256(digest_payload.encode("utf-8")).hexdigest(),
    "truncated": True,
}

print("status =", bounded_payload["status"])
print("summary_chars =", len(bounded_payload["summary"]))
print("original_chars =", bounded_payload["output_chars"])
print("artifact_refs =", len(bounded_payload["artifacts"]))
print("digest_chars =", len(bounded_payload["output_sha256"]))


status = output_too_large
summary_chars = 32
original_chars = 160
artifact_refs = 2
digest_chars = 64


**发生了什么**：输出越界成为显式状态。Lead 可以选择压缩、按需读取 Artifact，或向用户说明证据缺口。digest 只用来比较内容身份，不能恢复原文，也不是数字签名。

**动手修改**：只改变最后一个字符并重新计算 digest。确认 digest 改变，再说明完整内容应由 Artifact repository 保存，而不是靠 digest 保存。

## 实验 16：只记录任务终态和边界事实

`concept` · `baseline` · `delegation-ledger`

**运行前先预测**：审计记录只接收 task_id、status、context key、preview、长度和 digest。它会不会自动复制完整 Prompt、messages 或 Secret？

> 先在这里写下你的判断，再执行下一个代码单元。

In [16]:
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class DelegationRecord:
    task_id: str
    status: str
    context_keys: tuple[str, ...]
    summary_preview: str
    output_chars: int
    output_sha256: str


record = DelegationRecord(
    task_id="task-001",
    status=bounded_payload["status"],
    context_keys=("locale", "user_id"),
    summary_preview=bounded_payload["summary"][:16],
    output_chars=bounded_payload["output_chars"],
    output_sha256=bounded_payload["output_sha256"],
)
record_text = repr(record)

print("record_fields =", sorted(asdict(record)))
print("preview_chars =", len(record.summary_preview))
print("output_chars =", record.output_chars)
print("messages_recorded =", "主会话消息" in record_text)
print("secret_recorded =", "never-forward" in record_text)


record_fields = ['context_keys', 'output_chars', 'output_sha256', 'status', 'summary_preview', 'task_id']
preview_chars = 16
output_chars = 160
messages_recorded = False
secret_recorded = False


**发生了什么**：Delegation record 记录谁执行、终态如何、输入边界是否生效、输出身份是什么，但不复制主会话。它与业务 State、模型消息和完整 trace 属于不同数据边界。

**动手修改**：加入 tenant_id 和 error_code，但不要加入完整 exception。写下 retention 和 PII 删除策略应由哪一层负责。

## 实验 17：两个 specialist 各自创建临时 Agent，并排除主 messages

`migration` · `contrast` · `ephemeral-subagent`

**运行前先预测**：父上下文包含 locale、request_id、messages 和 auth_token。两个 specialist 的 ledger context_keys 会保留哪些？

> 先在这里写下你的判断，再执行下一个代码单元。

In [17]:
from mini_deerflow.subagents import (
    SubagentExecutor,
    SubagentRequest,
    build_demo_subagent_registry,
)


demo_registry = build_demo_subagent_registry()
demo_executor = SubagentExecutor(demo_registry, max_concurrency=2)
demo_requests = [
    SubagentRequest(
        task_id="demo-research",
        agent_name="research",
        description="研究 reducer",
        prompt="解释 reducer 的并行合并边界",
    ),
    SubagentRequest(
        task_id="demo-coding",
        agent_name="coding",
        description="设计 reducer 测试",
        prompt="给出防重复合并的测试建议",
    ),
]
demo_results = await demo_executor.dispatch_many(
    demo_requests,
    parent_context={
        "locale": "zh-CN",
        "request_id": "chapter-11",
        "messages": ["主会话不应转发"],
        "auth_token": "never-forward",
    },
)
demo_records = demo_executor.ledger.list_records()

print("capabilities =", demo_registry.describe())
print("statuses =", [result.status for result in demo_results])
print("summary_prefixes =", [result.summary[:4] for result in demo_results])
print("ledger_context_keys =", [record.context_keys for record in demo_records])
print("secret_in_ledger =", "never-forward" in repr(demo_records))


capabilities = (('research', '检索、比较并压缩证据'), ('coding', '分析 Python 接口并提出可测试实现'))
statuses = ['completed', 'completed']
summary_prefixes = ['研究摘要', '代码建议']
ledger_context_keys = [('locale', 'request_id'), ('locale', 'request_id')]
secret_in_ledger = False


**发生了什么**：Registry 只描述稳定名称、handler 和 policy。Executor 每次新建 `SubagentInvocation`，只投影 allowlist；built-in handler 使用 `checkpointer=False` 创建临时 Agent，不保留长期历史。

**动手修改**：在 registry 中增加 reviewer，并重复注册同名 specialist。记录重复名称为何必须在组合根启动时失败，而不能静默覆盖。

## 实验 18：task tool 隐藏 Registry、Semaphore 与 handler

`migration` · `contrast` · `delegation-boundary`

**运行前先预测**：模型调用 task 时，参数 Schema 中会暴露 executor、handler 或 auth_token 吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [18]:
from mini_deerflow.subagents import build_task_tool


task_tool = build_task_tool(demo_executor)
task_schema_fields = sorted(task_tool.tool_call_schema.model_fields)

print("tool_name =", task_tool.name)
print("schema_fields =", task_schema_fields)
print("executor_exposed =", "executor" in task_schema_fields)
print("auth_token_exposed =", "auth_token" in task_schema_fields)
print("max_concurrency_metadata =", task_tool.metadata["max_concurrency"])


tool_name = task
schema_fields = ['description', 'prompt', 'subagent_type', 'task_id']
executor_exposed = False
auth_token_exposed = False
max_concurrency_metadata = 2


**发生了什么**：模型只选择 specialist 并描述任务。Runtime Context、Registry、Semaphore 和 Ledger 都留在应用层，形成可测试且不能由模型绕过的执行接缝。

**动手修改**：向 args 添加未知字段。观察工具 Schema 如何拒绝或忽略它，并把期望行为写成测试，避免升级后静默变化。

## 实验 19：Lead 真实调用 task，再读取 ToolMessage 完成综合

`migration` · `contrast` · `delegation-boundary`

**运行前先预测**：一次完整 Agent loop 的消息顺序是什么？最终回答来自 specialist，还是 Lead 的第二次模型响应？

> 先在这里写下你的判断，再执行下一个代码单元。

In [19]:
import json

from langchain_core.messages import AIMessage, ToolMessage

from mini_deerflow.agents import create_lead_agent
from mini_deerflow.config import LeadAgentContext
from mini_deerflow.models import create_offline_model
from mini_deerflow.schemas import SubagentResult


lead_model = create_offline_model(
    [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "task",
                    "args": {
                        "task_id": "lead-task-1",
                        "description": "研究 checkpoint",
                        "prompt": "压缩成三条恢复原则",
                        "subagent_type": "research",
                    },
                    "id": "lead-call-1",
                    "type": "tool_call",
                }
            ],
        ),
        AIMessage(content="我已读取 Subagent 结果，并完成综合。"),
    ]
)
lead_agent = create_lead_agent(model=lead_model, tools=[task_tool])
lead_state = await lead_agent.ainvoke(
    {"messages": [{"role": "user", "content": "解释 checkpoint"}]},
    context=LeadAgentContext(
        user_id="learner",
        workspace_root="/tmp/lesson",
        auth_token="hidden",
    ),
)
tool_message = next(
    message for message in lead_state["messages"] if isinstance(message, ToolMessage)
)
delegated = SubagentResult.model_validate(json.loads(tool_message.content))

print("message_types =", [type(message).__name__ for message in lead_state["messages"]])
print("delegated_agent =", delegated.agent_name)
print("delegated_status =", delegated.status)
print("final_answer =", lead_state["messages"][-1].content)
print("secret_in_tool_message =", "hidden" in str(tool_message.content))


message_types = ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']
delegated_agent = research
delegated_status = completed
final_answer = 我已读取 Subagent 结果，并完成综合。
secret_in_tool_message = False


**发生了什么**：Lead 的第一次模型响应选择 task，工具返回结构化结果，第二次模型响应负责最终综合。控制权始终回到 Lead，这一点把 Subagent-as-tool 与 Handoff 区分开来。

**动手修改**：让模型同轮发出 research 和 coding 两个 task call。检查两条 ToolMessage，再说明并发上限由哪里执行。

## 实验 20：allowlist 过滤父上下文，并拒绝 secret-shaped 字段

`migration` · `contrast` · `context-projection`

**运行前先预测**：自定义 specialist 只允许 user_id 与 locale。handler 实际看到的 context 是什么？把 access_token 放进 allowlist 会发生什么？

> 先在这里写下你的判断，再执行下一个代码单元。

In [20]:
from mini_deerflow.subagents import (
    SubagentInvocation,
    SubagentOutput,
    SubagentRegistry,
    SubagentSpec,
)


observed_contexts: list[dict[str, object]] = []


async def inspect_context(invocation: SubagentInvocation) -> SubagentOutput:
    observed_contexts.append(invocation.context)
    return SubagentOutput(summary="context inspected")


context_executor = SubagentExecutor(
    SubagentRegistry(
        [
            SubagentSpec(
                name="inspector",
                description="检查输入投影",
                handler=inspect_context,
                allowed_context_fields=frozenset({"user_id", "locale"}),
            )
        ]
    )
)
context_result = await context_executor.dispatch(
    SubagentRequest(
        task_id="context-mini-001",
        agent_name="inspector",
        description="检查边界",
        prompt="只报告允许字段",
    ),
    parent_context={
        "user_id": "learner-11",
        "locale": "zh-CN",
        "messages": ["主历史"],
        "auth_token": "never-forward",
    },
)
try:
    SubagentSpec(
        name="unsafe",
        description="错误 policy",
        handler=inspect_context,
        allowed_context_fields=frozenset({"access_token"}),
    )
except ValueError as policy_error:
    secret_policy_rejected = "secret 字段" in str(policy_error)
else:
    secret_policy_rejected = False

print("observed_context_items =", sorted(observed_contexts[0].items()))
print("status =", context_result.status)
print("messages_visible =", "messages" in observed_contexts[0])
print("secret_policy_rejected =", secret_policy_rejected)


observed_context_items = [('locale', 'zh-CN'), ('user_id', 'learner-11')]
status = completed
messages_visible = False
secret_policy_rejected = True


**发生了什么**：Executor 按 spec 投影输入，`SubagentSpec` 还会拒绝形似 Secret 的 allowlist。字段名检查只是一道护栏；真实授权仍由工具和服务端执行。

**动手修改**：加入 sandbox_id 允许字段并重新运行。解释为什么传句柄比复制工作区内容更适合隔离边界。

## 实验 21：Mini DeerFlow 的 Semaphore 限制真实 handler 峰值

`migration` · `contrast` · `concurrency-control`

**运行前先预测**：Executor max_concurrency=2，提交 4 个请求。handler 的实际峰值和结果顺序是什么？

> 先在这里写下你的判断，再执行下一个代码单元。

In [21]:
mini_counter = {"active": 0, "peak": 0}


async def measured_handler(invocation: SubagentInvocation) -> SubagentOutput:
    mini_counter["active"] += 1
    mini_counter["peak"] = max(mini_counter["peak"], mini_counter["active"])
    await asyncio.sleep(0.01)
    mini_counter["active"] -= 1
    return SubagentOutput(summary=f"done:{invocation.prompt}")


mini_concurrency_executor = SubagentExecutor(
    SubagentRegistry(
        [SubagentSpec(name="worker", description="测量并发", handler=measured_handler)]
    ),
    max_concurrency=2,
)
mini_concurrency_results = await mini_concurrency_executor.dispatch_many(
    [
        SubagentRequest(
            task_id=f"mini-concurrency-{index}",
            agent_name="worker",
            description="并发实验",
            prompt=str(index),
        )
        for index in range(4)
    ]
)

print("peak_concurrency =", mini_counter["peak"])
print("statuses =", [result.status for result in mini_concurrency_results])
print("summaries =", [result.summary for result in mini_concurrency_results])


peak_concurrency = 2
statuses = ['completed', 'completed', 'completed', 'completed']
summaries = ['done:0', 'done:1', 'done:2', 'done:3']


**发生了什么**：同一 Executor 的所有 dispatch 都经过一个 Semaphore。`dispatch_many` 并发提交，`gather` 按请求顺序返回，所以结果顺序不依赖完成时序。

**动手修改**：把一个 handler 改成阻塞式 `time.sleep`。观察 event loop 受阻，并说明何时必须升级到进程、容器或远程 worker。

## 实验 22：Mini DeerFlow 将异常和 timeout 归一化为 SubagentResult

`migration` · `contrast` · `partial-failure`

**运行前先预测**：success、failure、timeout 同批提交。一个异常会不会取消另外两个结果？异常原文会不会完整写入 error？

> 先在这里写下你的判断，再执行下一个代码单元。

In [22]:
async def unstable_handler(invocation: SubagentInvocation) -> SubagentOutput:
    if invocation.prompt == "failure":
        raise RuntimeError("sensitive provider payload")
    if invocation.prompt == "timeout":
        await asyncio.sleep(0.05)
    return SubagentOutput(summary=f"ok:{invocation.prompt}")


mini_failure_executor = SubagentExecutor(
    SubagentRegistry(
        [SubagentSpec(name="unstable", description="故障注入", handler=unstable_handler)]
    ),
    max_concurrency=2,
    timeout_seconds=0.01,
)
mini_failure_results = await mini_failure_executor.dispatch_many(
    [
        SubagentRequest(
            task_id=f"mini-failure-{index}",
            agent_name="unstable",
            description="故障实验",
            prompt=value,
        )
        for index, value in enumerate(["success", "failure", "timeout"])
    ]
)

print("statuses =", [result.status for result in mini_failure_results])
print("success_summary =", mini_failure_results[0].summary)
print("failure_error =", mini_failure_results[1].error)
print("timeout_has_budget_message =", "执行预算" in (mini_failure_results[2].error or ""))


statuses = ['completed', 'failed', 'timed_out']
success_summary = ok:success
failure_error = RuntimeError: subagent handler failed
timeout_has_budget_message = True


**发生了什么**：Executor 负责归一化异常。它保留异常类型，丢弃可能包含敏感信息的供应商原文；timeout 单独表示预算耗尽，同批成功结果仍能交给 Lead。

**动手修改**：请求未知 agent_name。观察它返回 failed 而不是让 KeyError 越过 Lead 工具循环。

## 实验 23：Mini DeerFlow 同时限制摘要与 ArtifactRef 数量

`migration` · `contrast` · `output-budget`

**运行前先预测**：spec 允许 32 字符和 2 个 artifact。原始结果有 160 字符和 5 个引用，SubagentResult 会保留哪些审计信息？

> 先在这里写下你的判断，再执行下一个代码单元。

In [23]:
from mini_deerflow.schemas import ArtifactRef


async def verbose_handler(_: SubagentInvocation) -> SubagentOutput:
    return SubagentOutput(
        summary="证据" * 80,
        artifacts=[
            ArtifactRef(path=f"reports/{index}.md", media_type="text/markdown")
            for index in range(5)
        ],
    )


mini_budget_executor = SubagentExecutor(
    SubagentRegistry(
        [
            SubagentSpec(
                name="verbose",
                description="产生大输出",
                handler=verbose_handler,
                max_output_chars=32,
                max_artifacts=2,
            )
        ]
    )
)
mini_budget_result = await mini_budget_executor.dispatch(
    SubagentRequest(
        task_id="mini-large-output",
        agent_name="verbose",
        description="验证输出预算",
        prompt="返回大量证据",
    )
)

print("status =", mini_budget_result.status)
print("summary_chars =", len(mini_budget_result.summary))
print("original_chars =", mini_budget_result.output_chars)
print("artifact_refs =", len(mini_budget_result.artifacts))
print("digest_chars =", len(mini_budget_result.output_sha256 or ""))
print("truncated =", mini_budget_result.truncated)


status = output_too_large
summary_chars = 32
original_chars = 160
artifact_refs = 2
digest_chars = 64
truncated = True


**发生了什么**：Mini DeerFlow 把概念实验里的预算规则放进统一结果协议。完整大输出由 Sandbox/Artifact provider 保存，Lead 只在需要时读取。

**动手修改**：只让 artifact 数量越界、摘要不过界。确认状态仍是 output_too_large，并检查 error 明确指出哪项预算超限。

## 实验 24：Ledger 保存 context key、预览、长度与 digest

`migration` · `contrast` · `delegation-ledger`

**运行前先预测**：demo_executor 已执行多次研究和代码任务。Ledger 会保存完整 messages 和 Secret，还是只保存有界审计字段？

> 先在这里写下你的判断，再执行下一个代码单元。

In [24]:
ledger_records = demo_executor.ledger.list_records()
latest_record = ledger_records[-1]
rendered_records = repr(ledger_records)

print("record_count =", len(ledger_records))
print("latest_task_id =", latest_record.task_id)
print("latest_status =", latest_record.status)
print("latest_context_keys =", latest_record.context_keys)
print("summary_preview_chars =", len(latest_record.summary_preview))
print("digest_chars =", len(latest_record.output_sha256 or ""))
print("messages_in_ledger =", "主会话不应转发" in rendered_records)
print("secret_in_ledger =", "never-forward" in rendered_records)


record_count = 3
latest_task_id = lead-task-1
latest_status = completed
latest_context_keys = ('locale', 'request_id')
summary_preview_chars = 21
digest_chars = 64
messages_in_ledger = False
secret_in_ledger = False


**发生了什么**：同一个 demo executor 先执行两次直接委派，又通过 task tool 执行一次 Lead 委派。Ledger 只记录 task 终态和安全输入键，不替代 Checkpointer、模型历史或完整 trace。

**动手修改**：把 Ledger 替换为持久化 repository 接口草图。列出 tenant ownership、retention、PII policy 和事务边界，不要直接永久保存所有 Prompt。